# Jina Embeddings Retrieval Evaluation (T4 GPU)

Evaluates 4 Jina embedding models for Arabic WordNet entry retrieval using FAISS.

**Models**: jina-v5-nano (239M), jina-v5-small (677M), jina-v3 (570M), jina-v4 (3.8B)

**Requirements**: T4 GPU runtime (16GB VRAM), ~37MB data upload

## Setup

In [1]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Tesla T4, 15360 MiB
CUDA available: True
Device: Tesla T4


In [3]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.1 MB/s eta 0:00:00


In [4]:
!pip install -q condacolab
import condacolab
condacolab.install()  # this restarts the runtime automatically

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...


In [ ]:
!conda install -c conda-forge faiss-gpu -y
!pip install -q sentence-transformers pyyaml

In [26]:
!pip install -q peft einops Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 24.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Extract the data package from Google Drive
import zipfile, os
from pathlib import Path

gdrive_path = "/content/drive/MyDrive/Colab Notebooks/data/Jina Embeddings Retrieval Evaluation (T4 GPU)/jina_eval_data.zip"

if os.path.exists(gdrive_path):
    print(f"Extracting {gdrive_path}...")
    with zipfile.ZipFile(gdrive_path, "r") as z:
        z.extractall(".")

    os.chdir("retrieval_eval")
    !ls -la
else:
    print(f"Error: File not found at {gdrive_path}")

Extracting /content/drive/MyDrive/Colab Notebooks/data/Jina Embeddings Retrieval Evaluation (T4 GPU)/jina_eval_data.zip...
total 52
drwxr-xr-x   6 root root  4096 Mar 13 12:58 .
drwxr-xr-x   1 root root  4096 Mar 13 12:58 ..
-rw-r--r--   1 root root  7731 Mar 13 12:58 analysis.py
drwxr-xr-x   2 root root  4096 Mar 13 12:58 backends
drwxr-xr-x   3 root root  4096 Mar 13 12:58 export
drwxr-xr-x 287 root root 12288 Mar 13 12:58 prepared
-rw-r--r--   1 root root  3816 Mar 13 12:58 queries.py
-rw-r--r--   1 root root  7342 Mar 13 12:58 run_eval.py
drwxr-xr-x   2 root root  4096 Mar 13 12:58 runs


In [4]:
# Quick sanity check
from pathlib import Path
entries = list(Path("export/entries").glob("*.md"))
print(f"Entry files: {len(entries)}")
prepared = list(Path("prepared").iterdir())
print(f"Prepared synsets: {len([p for p in prepared if p.is_dir()])}")

Entry files: 1937
Prepared synsets: 285


## Run All 4 Jina Models

Each model: download from HuggingFace -> embed 1937 docs -> build FAISS index -> run 126 queries

In [7]:
# Run jina_v5_nano (239M params, 768 dims) - fastest
!python run_eval.py --backend jina_v5_nano --setup --num-synsets 206 --offset 0 --sleep 0

Running jina_v5_nano.setup()...
Loading jinaai/jina-embeddings-v5-text-nano on cuda...
modeling_eurobert.py: 48.9kB [00:00, 88.7MB/s]
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-nano:
- modeling_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-nano:
- modeling_eurobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
model.safetensors: 100% 424M/424M [00:03<00:00, 120MB/s]
/usr/local/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed

In [8]:
# Run jina_v5_small (677M params, 1024 dims)
!python run_eval.py --backend jina_v5_small --setup --num-synsets 206 --offset 0 --sleep 0

Running jina_v5_small.setup()...
Loading jinaai/jina-embeddings-v5-text-small on cuda...
modules.json: 100% 168/168 [00:00<00:00, 867kB/s]
config_sentence_transformers.json: 100% 282/282 [00:00<00:00, 1.58MB/s]
README.md: 14.2kB [00:00, 37.6MB/s]
custom_st.py: 3.93kB [00:00, 1.55MB/s]
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-small:
- custom_st.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
config.json: 100% 991/991 [00:00<00:00, 5.00MB/s]
configuration_jina_embeddings_v5.py: 100% 120/120 [00:00<00:00, 768kB/s]
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-embeddings-v5-text-small:
- configuration_jina_embeddings_v5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
mo

In [15]:
# Pre-download jina-v3's dependency (custom XLM-RoBERTa code)
from huggingface_hub import snapshot_download
snapshot_download("jinaai/xlm-roberta-flash-implementation", allow_patterns=["*.py", "*.json"])
snapshot_download("jinaai/jina-embeddings-v3")
print("jina-v3 model files ready")


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

jina-v3 model files ready


In [17]:
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

# Get path where snapshot was downloaded
dep_path = Path(snapshot_download("jinaai/xlm-roberta-flash-implementation"))

# Copy all .py files into the transformers module cache
cache_base = Path.home() / ".cache/huggingface/modules/transformers_modules/jinaai"
for commit_dir in cache_base.glob("xlm_hyphen_roberta_hyphen_flash_hyphen_implementation/*"):
    if commit_dir.is_dir():
        py_files = list(dep_path.glob("*.py"))
        for f in py_files:
            shutil.copy2(f, commit_dir / f.name)
        print(f"Copied {len(py_files)} .py files to {commit_dir.name[:12]}...")


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Copied 11 .py files to cd915ad56344...


In [20]:
import transformers.modeling_utils as _mu

_orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized

def _safe_mark(self, loading_info):
    if not hasattr(self, 'all_tied_weights_keys'):
        self.all_tied_weights_keys = {}
    return _orig_mark(self, loading_info)

_mu.PreTrainedModel.mark_tied_weights_as_initialized = _safe_mark
print("Patched for jina-v3 compatibility")


Patched for jina-v3 compatibility


In [22]:
# Patch _jina_common.py to fix jina-v3 transformers compatibility
path = "backends/_jina_common.py"
with open(path, "r") as f:
    content = f.read()

patch = '''
# -- jina-v3 compat: fix missing all_tied_weights_keys --
import transformers.modeling_utils as _mu
_orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
def _safe_mark(self, loading_info):
    if not hasattr(self, 'all_tied_weights_keys'):
        self.all_tied_weights_keys = {}
    return _orig_mark(self, loading_info)
_mu.PreTrainedModel.mark_tied_weights_as_initialized = _safe_mark
'''

content = content.replace(
    "from sentence_transformers import SentenceTransformer  # noqa: E402",
    "from sentence_transformers import SentenceTransformer  # noqa: E402\n" + patch
)

with open(path, "w") as f:
    f.write(content)
print("Patched _jina_common.py on disk")


Patched _jina_common.py on disk


In [23]:
# Run jina_v3 (570M params, 1024 dims)
!python run_eval.py --backend jina_v3 --setup --num-synsets 206 --offset 0 --sleep 0

Running jina_v3.setup()...
Loading jinaai/jina-embeddings-v3 on cuda...
`torch_dtype` is deprecated! Use `dtype` instead!
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch

In [28]:
!pip install -q --upgrade transformers


In [30]:
# Check actual version
!python -c "import transformers; print('transformers:', transformers.__version__)"

# Patch jina-v4's cached qwen2_5_vl.py to handle missing SlidingWindowCache
from pathlib import Path

qwen_file = Path("/root/.cache/huggingface/modules/transformers_modules/jinaai/"
                 "jina_hyphen_embeddings_hyphen_v4/"
                 "737fa5c46f0262ceba4a462ffa1c5bcf01da416f/qwen2_5_vl.py")

content = qwen_file.read_text()
old = "from transformers.cache_utils import Cache, DynamicCache, SlidingWindowCache, StaticCache"
new = """try:
    from transformers.cache_utils import Cache, DynamicCache, SlidingWindowCache, StaticCache
except ImportError:
    from transformers.cache_utils import Cache, DynamicCache, StaticCache
    SlidingWindowCache = DynamicCache  # fallback alias"""

content = content.replace(old, new)
qwen_file.write_text(content)
print("Patched qwen2_5_vl.py for SlidingWindowCache import")


transformers: 5.3.0
Patched qwen2_5_vl.py for SlidingWindowCache import


In [ ]:
# Run jina_v4 (3.8B params, 2048 dims, fp16)
!python run_eval.py --backend jina_v4 --setup --num-synsets 206 --offset 0 --sleep 0

## Analysis

In [32]:
# Run analysis for all backends
for backend in ["jina_v5_nano", "jina_v5_small", "jina_v3", "jina_v4"]:
    print(f"\n{'='*60}")
    print(f"  {backend}")
    print(f"{'='*60}")
    !python analysis.py --backend {backend}


  jina_v5_nano
# Jina V5 Nano Retrieval Evaluation Report

## Overview

- **Total queries evaluated:** 126
- **Skipped (0 GT overlap):** 489
- **Errors:** 0
- **Query types:** arabic_lemma, definition_keyword

## Metrics by Query Type

| Query Type | N | Recall@10 | Recall@25 | Recall@50 | P@10 | MRR | Avg GT |
|------------|---|-----------|-----------|-----------|------|-----|--------|
| arabic_lemma | 63 | 95.5% | 97.5% | 97.9% | 17.8% | 0.928 | 1.9 |
| definition_keyword | 63 | 48.3% | 62.2% | 64.9% | 8.3% | 0.412 | 1.9 |
| **Overall** | 126 | 71.9% | 79.8% | 81.4% | 13.0% | 0.670 | — |

## Interpretation

- **Recall@K** = fraction of ground-truth entries found in top-K results.
  SQL baseline has 100% recall (evidence.json was generated from SQL).
- **Precision@10** = fraction of top-10 results that are relevant.
- **MRR** = mean reciprocal rank of first relevant result (1.0 = always first).
- **Avg GT** = average number of ground truth entries per query (filtered to entries we up

## Download Results

In [33]:
# Package all results and save to Google Drive
import zipfile, os
from pathlib import Path

# Define output directory
output_dir = Path("/content/drive/MyDrive/Colab Notebooks/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

results_zip = output_dir / "jina_eval_results.zip"

with zipfile.ZipFile(results_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for backend in ["jina_v5_nano", "jina_v5_small", "jina_v3", "jina_v4"]:
        run_dir = Path("runs") / backend
        if not run_dir.exists():
            print(f"  Skipping {backend} (no results)")
            continue
        for f in run_dir.iterdir():
            # Skip large FAISS index files - we only need JSON + report
            if f.suffix == ".index":
                continue
            z.write(f, f"runs/{backend}/{f.name}")
            print(f"  Added: runs/{backend}/{f.name}")

print(f"\nResults saved to: {results_zip} ({os.path.getsize(results_zip)/1024:.0f} KB)")

  Added: runs/jina_v5_nano/report.md
  Added: runs/jina_v5_nano/config.json
  Added: runs/jina_v5_nano/filenames.json
  Added: runs/jina_v5_nano/retrieval_results.json
  Added: runs/jina_v5_small/report.md
  Added: runs/jina_v5_small/config.json
  Added: runs/jina_v5_small/filenames.json
  Added: runs/jina_v5_small/retrieval_results.json
  Added: runs/jina_v3/report.md
  Added: runs/jina_v3/config.json
  Added: runs/jina_v3/filenames.json
  Added: runs/jina_v3/retrieval_results.json

Results saved to: /content/drive/MyDrive/Colab Notebooks/outputs/jina_eval_results.zip (451 KB)
